In [1]:
import sys
from transformers import BertTokenizerFast, BertModel
import spacy
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch
import os
import seaborn as sns

from data import *
from model import *
from utils import *


/usr/local/anaconda3.2024/bin/python


In [5]:
os.environ["TOKENIZERS_PARALLELISM"] = "true"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Khởi tạo tokenizer và model BERT
tokenizer_bert = BertTokenizerFast.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")

# Thêm token đặc biệt nếu chưa có
special_tokens = {"additional_special_tokens": ["[E1]", "[/E1]", "[E2]", "[/E2]"]}
num_added_toks = tokenizer_bert.add_special_tokens(special_tokens)
bert.resize_token_embeddings(len(tokenizer_bert))  # Resize embedding BERT
bert = bert.to(device)



# Tokenizer và DataLoader
spacy_nlp = spacy.load('en_core_web_sm')
# Cập nhật tokenizer trong spaCy
spacy_nlp.tokenizer = create_custom_tokenizer(spacy_nlp, special_tokens['additional_special_tokens'])

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [22]:
data_source = "/data/ddao/TRE/data/raw_data/i2b2_2012/data_processed/"
data_train_path = data_source + "train_merged_full.csv"
data_test_path = data_source + "test_merged_full.csv"


train_df = pd.read_csv(data_train_path)
test_df = pd.read_csv(data_test_path)

# Mã hóa nhãn
label_encoder = LabelEncoder()
train_df['label_encoded'] = label_encoder.fit_transform(train_df['label'])
test_df['label_encoded'] = label_encoder.transform(test_df['label'])  # Ánh xạ giống tập huấn luyện
num_classes = len(label_encoder.classes_)
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print(f"Label Mapping: {label_mapping}")
relations = list(label_mapping.keys())


validation_size = 0.15
random_state = 42
train_df, valid_df = train_test_split(train_df, test_size=validation_size, random_state=random_state)

print("train_df:", train_df.shape)
print("valid_df:", valid_df.shape)
print("test_df:", test_df.shape)

print("Create dataloader:")
batch_size = 64
window_size = 128
MAX_LENGTH = 256
shuffle=False
print("batch_size: ", batch_size)
print("window_size: ", window_size)
print("MAX_LENGTH: ", MAX_LENGTH)

Label Mapping: {'AFTER': 0, 'BEFORE': 1, 'OVERLAP': 2}
train_df: (28305, 12)
valid_df: (4995, 12)
test_df: (27530, 12)
Create dataloader:
batch_size:  64
window_size:  128
MAX_LENGTH:  256


In [24]:
train_dataloader_path =  f"/data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/train_dataloader_ws_{window_size}.pt"
validation_dataloader_path = f"/data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/validation_dataloader_ws_{window_size}.pt"
test_dataloader_path = f"/data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/test_dataloader_ws_{window_size}.pt"

In [23]:
train_dataloader = torch.load(train_dataloader_path)
validation_dataloader = torch.load(validation_dataloader_path)
test_dataloader = torch.load(test_dataloader_path)

In [24]:
# ============================================================
# Hyperparameters
# ============================================================
input_dim        = 768
hidden_ffn_dim   = 512
dropout          = 0.5
top_k            = 10
num_heads        = 8

num_epochs       = 20
eval_step        = 1
LEARNING_RATE    = 1e-5

print(f"hidden_ffn_dim: {hidden_ffn_dim}")
print(f"top_k: {top_k}")
print(f"num_heads: {num_heads}")

# ============================================================
# Save path
# ============================================================
save_path = (
    f"/data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/"
    f"FTBERT_I2B2_V2_MHA_{num_heads}_TOP_{top_k}_FFN_{hidden_ffn_dim}.pth"
)

# ============================================================
# Create model, criterion, optimizer
# ============================================================
print("\nCreate model:")
model = TemporalRelationModelV2(
    bert=bert,
    tokenizer=tokenizer_bert,
    input_dim=input_dim,
    hidden_ffn_dim=hidden_ffn_dim,
    num_heads=num_heads,
    dropout=dropout,
    top_k=top_k,
    num_classes=num_classes,
    device=device
).to(device)

print("Device:", device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print("--" * 20)

hidden_ffn_dim: 512
top_k: 10
num_heads: 8

Create model:
Device: cuda
----------------------------------------


/home/prof/ddao/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


# Analyze result

In [25]:
print("Load pretrained model:")
checkpoint = torch.load(save_path)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
epoch = checkpoint['epoch']
loss = checkpoint['loss']
print(f"Model saved at epoch {epoch}, loss = {loss:.4f}")

Load pretrained model:
Model saved at epoch 19, loss = 0.0273


In [ ]:
top_k = 10

In [26]:
cm, f1, p, r, precisions, recalls, f1s, all_predictions, all_labels, logs = evaluate(train_dataloader,
             model,
             device,
             top_k=top_k,
             return_preds=True,
             return_topk_tokens=True,
             return_logs=True,
             is_I2B2=True)

Evaluating: 100%|██████████| 443/443 [05:05<00:00,  1.45it/s]

Evaluation finished in 305.12 seconds.
Accuracy: 0.9996, F1 (weighted): 0.9996, P: 0.9996, R: 0.9996


In [27]:
id2label = {v:k for k, v in label_mapping.items()}

In [28]:
def make_token(token, vocab=None):
    """
    Chuẩn hóa 1 token (có thể là spacy.Token hoặc dict) thành dict dễ đọc.
    """
    def to_str(val):
        if val is None:
            return ""
        if isinstance(val, str):
            return val
        if vocab is not None:
            try:
                return vocab.strings[val]
            except Exception:
                return str(val)
        return str(val)

    # nếu là dict
    if isinstance(token, dict):
        return {
            "idx": token.get("idx", ""),
            "text": token.get("text", ""),
            "lemma": to_str(token.get("lemma", "")),
            "pos": to_str(token.get("pos", "")),
            "tag": to_str(token.get("tag", "")),
            "dep": to_str(token.get("dep", "")),
            "morph": str(token.get("morph", ""))
        }
    else:
        # giả sử là spacy.Token
        return {
            "text": getattr(token, "text", ""),
            "idx": getattr(token, "idx", ""),
            "lemma": to_str(getattr(token, "lemma_", getattr(token, "lemma", ""))),
            "pos": to_str(getattr(token, "pos_", getattr(token, "pos", ""))),
            "tag": to_str(getattr(token, "tag_", getattr(token, "tag", ""))),
            "dep": to_str(getattr(token, "dep_", getattr(token, "dep", ""))),
            "morph": str(getattr(token, "morph", ""))
        }


def build_analysis_dataset(all_labels, all_predictions, logs, id2label, vocab=None, entity_pairs=None):
    """
    Tạo dataset phân tích từ kết quả evaluate và logs, có bổ sung is_intra.
    
    Args:
        all_labels: list[int] ground truth labels
        all_predictions: list[int] predicted labels
        logs: list[dict] (từ model khi return_topk_tokens=True)
        id2label: dict {label_id: label_name}
        vocab: spacy.vocab.Vocab để decode id sang string (nếu có)
        entity_pairs: list[list] (mỗi dòng từ df.values.tolist()) chứa thông tin entity + context + is_intra

    Returns:
        data: list[dict] dạng JSON phân tích, có thêm is_intra
    """
    data = []
    N = len(all_labels)

    for i in range(N):
        gold = id2label[all_labels[i]]
        pred = id2label[all_predictions[i]]
        log = logs[i]
        
        # Lấy is_intra từ entity_pairs (giả sử đã có cột is_intra ở vị trí cuối cùng)
        is_intra = entity_pairs[i][-1]  # cột cuối là is_intra
        context = entity_pairs[i][9]    # context vẫn ở cột 9 như cũ

        entry = {
            "e1_text": entity_pairs[i][6],
            "e2_text": entity_pairs[i][7],
            "e1_span": (entity_pairs[i][2], entity_pairs[i][4]),
            "e2_span": (entity_pairs[i][3], entity_pairs[i][5]),
            "context": context,
            "relation": gold,
            "prediction": pred,
            "is_intra": is_intra,
            "e1_tokens": [make_token(tok, vocab) for tok in log.get("e1_tokens", [])],
            "e2_tokens": [make_token(tok, vocab) for tok in log.get("e2_tokens", [])],
            "e1_topk": [make_token(tok, vocab) for tok in log.get("e1_topk_tokens", [])],
            "e2_topk": [make_token(tok, vocab) for tok in log.get("e2_topk_tokens", [])],
            "p_topk": [make_token(tok, vocab) for tok in log.get("pair_topk_tokens", [])],
            "e1_scores": log.get("e1_scores", []),
            "e2_scores": log.get("e2_scores", []),
            "p_scores": log.get("pair_scores", []),
        }
        data.append(entry)
    return data


def add_is_intra_batch(df, batch_size=64, n_process=None):
    """
    Xác định 2 entity cùng câu hay không sử dụng SpaCy pipe theo batch.
    Args:
        df: DataFrame gốc, có cột 'text', 'entity1_start', 'entity2_start'
        batch_size: số lượng câu mỗi batch
        n_process: số process để song song hóa
    Returns:
        df mới có cột 'is_intra' (1 = cùng câu, 0 = khác câu)
    """
    if n_process is None:
        from multiprocessing import cpu_count
        n_process = max(1, cpu_count() - 1)

    texts = df['text'].tolist()
    e1_starts = df['entity1_start'].tolist()
    e2_starts = df['entity2_start'].tolist()
    
    is_intra_list = []

    # Sử dụng nlp.pipe với batch và n_process
    for doc, e1_start, e2_start in zip(
        spacy_nlp.pipe(texts, batch_size=batch_size, n_process=n_process, disable=["ner", "tagger", "lemmatizer"]),
        e1_starts,
        e2_starts
    ):
        # Xác định sentence chứa entity
        e1_sent_id = e2_sent_id = None
        for i, sent in enumerate(doc.sents):
            if sent.start_char <= e1_start < sent.end_char:
                e1_sent_id = i
            if sent.start_char <= e2_start < sent.end_char:
                e2_sent_id = i

        is_intra = 1 if e1_sent_id is not None and e2_sent_id is not None and e1_sent_id == e2_sent_id else 0
        is_intra_list.append(is_intra)
    
    df['is_intra'] = is_intra_list
    return df


def create_analysis_dataframe(data):
    """
    Tạo DataFrame phân tích từ dataset JSON (trả về từ build_analysis_dataset)
    
    Mỗi token (top-k hoặc thực thể) là một dòng, có đầy đủ:
        - entity_type: e1 / e2 / pair
        - token, lemma, pos, tag, dep, morph
        - attention_score (nếu có)
        - prediction, relation (label)
        - is_intra: 1 nếu 2 entity cùng sentence, 0 nếu khác
    """
    rows = []

    for entry in data:
        # e1_topk
        e1_topk = entry.get("e1_topk", [])
        e1_scores = entry.get("e1_scores", [])
        for tok, score in zip(e1_topk, e1_scores):
            rows.append({
                "entity_type": "e1",
                "token": tok["text"],
                "lemma": tok.get("lemma", tok["text"]),
                "pos": tok.get("pos", None),
                "tag": tok.get("tag", None),
                "dep": tok.get("dep", None),
                "morph": tok.get("morph", None),
                "attention_score": score,
                "prediction": entry["prediction"],
                "relation": entry["relation"],
                "is_intra": entry.get("is_intra", 1)
            })
        # e2_topk
        e2_topk = entry.get("e2_topk", [])
        e2_scores = entry.get("e2_scores", [])
        for tok, score in zip(e2_topk, e2_scores):
            rows.append({
                "entity_type": "e2",
                "token": tok["text"],
                "lemma": tok.get("lemma", tok["text"]),
                "pos": tok.get("pos", None),
                "tag": tok.get("tag", None),
                "dep": tok.get("dep", None),
                "morph": tok.get("morph", None),
                "attention_score": score,
                "prediction": entry["prediction"],
                "relation": entry["relation"],
                "is_intra": entry.get("is_intra", 1)
            })
        # pair_topk
        p_topk = entry.get("p_topk", [])
        p_scores = entry.get("p_scores", [])
        for tok, score in zip(p_topk, p_scores):
            rows.append({
                "entity_type": "pair",
                "token": tok["text"],
                "lemma": tok.get("lemma", tok["text"]),
                "pos": tok.get("pos", None),
                "tag": tok.get("tag", None),
                "dep": tok.get("dep", None),
                "morph": tok.get("morph", None),
                "attention_score": score,
                "prediction": entry["prediction"],
                "relation": entry["relation"],
                "is_intra": entry.get("is_intra", 1)
            })

    df = pd.DataFrame(rows)
    return df


def summarize_by_intra(df_analysis):
    """
    Thống kê POS / dep / morph phân theo entity_type và is_intra
    """
    # POS distribution
    pos_counts = df_analysis.groupby(['entity_type','is_intra'])['pos'] \
                            .value_counts(normalize=True).unstack(fill_value=0)
    print("POS distribution by entity_type and is_intra:")
    print(pos_counts)

    plt.figure(figsize=(14,6))
    sns.heatmap(pos_counts, annot=True, cmap='Blues')
    plt.title('POS Distribution by Entity Type and is_intra')
    plt.ylabel('Entity Type + is_intra')
    plt.xlabel('POS Tag')
    plt.show()

    # Dependency distribution
    dep_counts = df_analysis.groupby(['entity_type','is_intra'])['dep'] \
                            .value_counts(normalize=True).unstack(fill_value=0)
    print("Dependency distribution by entity_type and is_intra:")
    print(dep_counts)

    plt.figure(figsize=(14,6))
    sns.heatmap(dep_counts, annot=True, cmap='Greens')
    plt.title('Dependency Distribution by Entity Type and is_intra')
    plt.ylabel('Entity Type + is_intra')
    plt.xlabel('Dependency Tag')
    plt.show()

    # Morphology distribution
    morph_counts = df_analysis.groupby(['entity_type','is_intra'])['morph'] \
                              .value_counts(normalize=True).unstack(fill_value=0)
    print("Morphology distribution by entity_type and is_intra:")
    print(morph_counts)

    plt.figure(figsize=(14,6))
    sns.heatmap(morph_counts, annot=True, cmap='Oranges')
    plt.title('Morphology Distribution by Entity Type and is_intra')
    plt.ylabel('Entity Type + is_intra')
    plt.xlabel('Morphological Features')
    plt.show()


def analyze_correct_prediction_intra(df_analysis):
    # Lọc các token cùng câu và prediction đúng
    df_intra = df_analysis[df_analysis['is_intra']==1]
    df_correct = df_intra[df_intra['prediction'] == df_intra['relation']]

    print("Số token intra-sentence và prediction đúng:", len(df_correct))

    # POS distribution
    pos_dist = df_correct.groupby('entity_type')['pos'].value_counts(normalize=True).unstack(fill_value=0)
    print("POS distribution (intra, correct prediction):")
    print(pos_dist)

    plt.figure(figsize=(12,5))
    sns.heatmap(pos_dist, annot=True, cmap='Blues')
    plt.title('POS Distribution by Entity Type (Intra-sentence, Correct Prediction)')
    plt.ylabel('Entity Type')
    plt.xlabel('POS Tag')
    plt.show()

    # Dependency distribution
    dep_dist = df_correct.groupby('entity_type')['dep'].value_counts(normalize=True).unstack(fill_value=0)
    print("Dependency distribution (intra, correct prediction):")
    print(dep_dist)

    plt.figure(figsize=(12,5))
    sns.heatmap(dep_dist, annot=True, cmap='Greens')
    plt.title('Dependency Distribution by Entity Type (Intra-sentence, Correct Prediction)')
    plt.ylabel('Entity Type')
    plt.xlabel('Dependency Tag')
    plt.show()

    # Morphology distribution
    morph_dist = df_correct.groupby('entity_type')['morph'].value_counts(normalize=True).unstack(fill_value=0)
    print("Morphology distribution (intra, correct prediction):")
    print(morph_dist)

    plt.figure(figsize=(12,5))
    sns.heatmap(morph_dist, annot=True, cmap='Oranges')
    plt.title('Morphology Distribution by Entity Type (Intra-sentence, Correct Prediction)')
    plt.ylabel('Entity Type')
    plt.xlabel('Morphological Features')
    plt.show()

def analyze_pos_dep_morph_by_relation(df_analysis):
    """
    Phân tích POS / dep / morph theo relation, chỉ xem prediction đúng và is_intra = 1
    """
    # Lọc token cùng câu và prediction đúng
    df_intra_correct = df_analysis[(df_analysis['is_intra']==1) &
                                   (df_analysis['prediction'] == df_analysis['relation'])]

    print("Số token intra-sentence với prediction đúng:", len(df_intra_correct))

    relations = df_intra_correct['relation'].unique()
    entity_types = df_intra_correct['entity_type'].unique()

    for rel in relations:
        print(f"\n=== Relation: {rel} ===")
        df_rel = df_intra_correct[df_intra_correct['relation']==rel]

        # POS distribution
        pos_dist = df_rel.groupby('entity_type')['pos'].value_counts(normalize=True).unstack(fill_value=0)
        print("POS distribution:")
        print(pos_dist)

        plt.figure(figsize=(12,4))
        sns.heatmap(pos_dist, annot=True, cmap='Blues')
        plt.title(f'POS Distribution - Relation={rel} (Intra-sentence, Correct Prediction)')
        plt.ylabel('Entity Type')
        plt.xlabel('POS Tag')
        plt.show()

        # Dependency distribution
        dep_dist = df_rel.groupby('entity_type')['dep'].value_counts(normalize=True).unstack(fill_value=0)
        print("Dependency distribution:")
        print(dep_dist)

        plt.figure(figsize=(12,4))
        sns.heatmap(dep_dist, annot=True, cmap='Greens')
        plt.title(f'Dependency Distribution - Relation={rel} (Intra-sentence, Correct Prediction)')
        plt.ylabel('Entity Type')
        plt.xlabel('Dependency Tag')
        plt.show()

        # Morphology distribution
        morph_dist = df_rel.groupby('entity_type')['morph'].value_counts(normalize=True).unstack(fill_value=0)
        print("Morphology distribution:")
        print(morph_dist)

        plt.figure(figsize=(12,4))
        sns.heatmap(morph_dist, annot=True, cmap='Oranges')
        plt.title(f'Morphology Distribution - Relation={rel} (Intra-sentence, Correct Prediction)')
        plt.ylabel('Entity Type')
        plt.xlabel('Morphological Features')
        plt.show()
        
def pos_distribution(df_analysis):
    pos_counts = df_analysis.groupby('entity_type')['pos'].value_counts(normalize=True).unstack(fill_value=0)
    print("POS distribution per entity_type:")
    print(pos_counts)
    
    # Vẽ heatmap POS distribution
    plt.figure(figsize=(12,6))
    sns.heatmap(pos_counts, annot=True, cmap='Blues')
    plt.title('POS Distribution per Entity Type')
    plt.ylabel('Entity Type')
    plt.xlabel('POS Tag')
    plt.show()

def dep_distribution(df_analysis):
    dep_counts = df_analysis.groupby('entity_type')['dep'].value_counts(normalize=True).unstack(fill_value=0)
    print("Dependency distribution per entity_type:")
    print(dep_counts)
    
    # Vẽ heatmap Dependency distribution
    plt.figure(figsize=(12,6))
    sns.heatmap(dep_counts, annot=True, cmap='Greens')
    plt.title('Dependency Distribution per Entity Type')
    plt.ylabel('Entity Type')
    plt.xlabel('Dependency Tag')
    plt.show()
    
def morph_distribution(df_analysis):
    morph_counts = df_analysis.groupby('entity_type')['morph'].value_counts(normalize=True).unstack(fill_value=0)
    print("Morphology distribution per entity_type:")
    print(morph_counts)
    
    # Heatmap Morphology distribution
    plt.figure(figsize=(12,6))
    sns.heatmap(morph_counts, annot=True, cmap='Oranges')
    plt.title('Morphology Distribution per Entity Type')
    plt.ylabel('Entity Type')
    plt.xlabel('Morphological Features')
    plt.show()

In [32]:
json_path = "/data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/data4analyze.json"

with open(json_path, "r", encoding="utf-8") as f:
    d4A = json.load(f)

In [33]:
df_expanded = add_is_intra_batch(train_df, batch_size=64)
d4A = build_analysis_dataset(all_labels,
                             all_predictions,
                             logs,
                             id2label,
                             entity_pairs=df_expanded.values.tolist(),
                             vocab=spacy_nlp.vocab)  # giúp decode id → string)

# Write dictionary to JSON file
with open(json_path, "w") as json_file:
    json.dump(d4A, json_file, indent=4)


In [36]:
df_analysis = create_analysis_dataframe(d4A)
df_analysis.to_json("/data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/train_D4A.json")
# df_analysis = pd.read_json("/data/ddao/TRE/pretrained_models/BERT_ATT_TOP_K_FFN/I2B2/train_D4A.json")

In [37]:
pos_distribution(df_analysis)
dep_distribution(df_analysis)
morph_distribution(df_analysis)
summarize_by_intra(df_analysis)
analyze_correct_prediction_intra(df_analysis)
analyze_pos_dep_morph_by_relation(df_analysis)